# ENEMDU 2021-2025 — Pipeline Medallón (Bronze → Silver → Gold)

**Proyecto:** UCUENCA-SABE · MVP Dashboard de Empleabilidad

**Cambio clave:** Descarga **inteligente**. Si ya existe Bronze localmente, no descarga de nuevo. Opción `force_download=True` para reforzar.

---

## 1. Control de descarga — Caché local

**Lógica:**
1. Revisa si los datos ya existen en `data/bronze/externas/enemdu/`
2. Si existen todos los años esperados → **salta la descarga**
3. Si faltan años o `force_download=True` → descarga desde Kaggle
4. Guarda un timestamp de cuándo se descargó (para auditoría)

In [ ]:
import sys
from pathlib import Path
import json
from datetime import datetime

# Inyectar la raíz del proyecto al sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG, get_external_bronze_dir
from src.ingestion.kaggle_downloader import download_and_organize_enemdu

print(f"✅ Proyecto cargado correctamente: {CONFIG['project']['name']}")
print(f"📁 Ruta raíz: {PROJECT_ROOT}")
print(f"📊 Sources disponibles: {list(CONFIG['sources'].keys())}")

## 2. Función de control inteligente de descarga

In [ ]:
def check_bronze_cache(
    years_expected: range = range(2021, 2026),
    force_download: bool = False
) -> dict:
    """
    Verifica si Bronze ya existe localmente.
    
    Parámetros:
    -----------
    years_expected : range
        Años que se espera tener (default: 2021-2025)
    force_download : bool
        Si True, ignora el caché y descarga de todas formas
    
    Retorna:
    --------
    dict con claves:
      - 'existe': bool, ¿datos locales completos?
      - 'años_locales': list[int], años presentes
      - 'años_faltantes': list[int], años que no hay
      - 'tamaño_mb': float, peso total de Bronze
      - 'última_descarga': str (ISO), timestamp del manifest
      - 'acción': str, lo que se va a hacer
    """
    bronze_dir = get_external_bronze_dir() / "enemdu"
    csv_dir = bronze_dir / "microdatos_csv"
    manifest_path = bronze_dir / ".manifest.json"
    
    # Detectar años presentes en local
    if csv_dir.exists():
        import re
        años_locales = sorted(
            int(m.group()) for f in csv_dir.glob("*.csv")
            if (m := re.search(r"(?:19|20)\d{2}", f.stem))
        )
    else:
        años_locales = []
    
    años_faltantes = [a for a in years_expected if a not in años_locales]
    existe_completo = len(años_faltantes) == 0 and len(años_locales) > 0
    
    # Leer timestamp de última descarga (si existe)
    última_descarga = None
    if manifest_path.exists():
        try:
            with open(manifest_path) as f:
                manifest = json.load(f)
                última_descarga = manifest.get("última_descarga")
        except:
            pass
    
    # Calcular tamaño total
    tamaño_mb = 0
    if csv_dir.exists():
        tamaño_mb = sum(f.stat().st_size for f in csv_dir.rglob("*.csv")) / 1e6
    
    # Decidir acción
    if force_download:
        acción = "DESCARGANDO (forzado por usuario)"
    elif existe_completo:
        acción = "✅ SALTANDO — datos completos en caché"
    elif años_locales:
        acción = f"DESCARGANDO — faltan {len(años_faltantes)} años"
    else:
        acción = "DESCARGANDO — Bronze vacío"
    
    return {
        "existe": existe_completo,
        "años_locales": años_locales,
        "años_faltantes": años_faltantes,
        "tamaño_mb": tamaño_mb,
        "última_descarga": última_descarga,
        "acción": acción,
        "force_download": force_download,
    }

# Verificar estado
estado = check_bronze_cache(years_expected=range(2021, 2026), force_download=False)
print("\n" + "="*70)
print("📦 ESTADO DE CACHÉ LOCAL")
print("="*70)
print(f"  Años locales:      {estado['años_locales'] if estado['años_locales'] else 'ninguno'}")
if estado['años_faltantes']:
    print(f"  Años faltantes:    {estado['años_faltantes']}")
print(f"  Tamaño en disco:   {estado['tamaño_mb']:.1f} MB")
if estado['última_descarga']:
    print(f"  Última descarga:   {estado['última_descarga']}")
print(f"\n  ⚡ ACCIÓN: {estado['acción']}")
print("="*70)

## 3. Descarga condicionada

In [ ]:
# ⚙️ PARÁMETRO: cambiar a True si quieres forzar descarga nuevamente
FORCE_DOWNLOAD = False

estado = check_bronze_cache(years_expected=range(2021, 2026), force_download=FORCE_DOWNLOAD)

if estado["existe"] and not estado["force_download"]:
    print(f"\n✅ Bronze ya existe localmente ({estado['tamaño_mb']:.1f} MB)")
    print(f"   Años completos: {estado['años_locales']}")
    print(f"   Última descarga: {estado['última_descarga'] or 'desconocida'}")
    print("\n⏭️  Saltando descarga de Kaggle (ahorro: ~9 minutos)")
    inventario = None  # placeholder
else:
    print(f"\n🚀 Iniciando descarga desde Kaggle...")
    print(f"   (esto tomará ~9 minutos)\n")
    try:
        inventario = download_and_organize_enemdu()
        
        # Guardar timestamp de descarga
        bronze_dir = get_external_bronze_dir() / "enemdu"
        manifest_path = bronze_dir / ".manifest.json"
        manifest = {
            "última_descarga": datetime.now().isoformat(),
            "años": list(range(2021, 2026)),
            "fuente": "https://www.kaggle.com/datasets/kmichelle/enemdu-ecuador-microdatos-anuales-personas"
        }
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2, default=str)
        
        print(f"\n✅ Descarga completada y caché actualizado.")
    except Exception as e:
        print(f"❌ Error: {e}")

## 4. Verificación post-descarga

Confirmar que todos los años están presentes.

In [ ]:
# Re-chequear después de la descarga (si ocurrió)
estado_final = check_bronze_cache(years_expected=range(2021, 2026), force_download=False)

print("\n" + "="*70)
print("✅ VERIFICACIÓN FINAL")
print("="*70)
print(f"  Años disponibles:  {estado_final['años_locales']}")
print(f"  Total en disco:    {estado_final['tamaño_mb']:.1f} MB")
print(f"  Completo:          {'SÍ ✅' if estado_final['existe'] else 'NO ⚠️'}")
print("="*70)

---

## 5. Ejemplo: Forzar descarga manual

Si necesitas refrescar los datos (ej., el INEC publicó nuevos microdatos), cambia `FORCE_DOWNLOAD = True` en la celda anterior y re-ejecuta.

In [ ]:
def manual_download_if_needed(años: list[int] = None):
    """
    Descarga solo si falta algún año específico.
    Útil para actualizar parcialmente.
    
    Ejemplo:
      manual_download_if_needed(años=[2025])  # Re-descarga 2025 solamente
      manual_download_if_needed()  # Re-descarga TODO
    """
    if años is None:
        años = list(range(2021, 2026))
    
    estado = check_bronze_cache(years_expected=range(2021, 2026), force_download=True)
    print(f"🔄 Forzando descarga de todos los años: {estado['años_faltantes'] or 'todo completo'}...")
    download_and_organize_enemdu()
    print("✅ Listo")

# Descomenta la siguiente línea si necesitas forzar descarga:
# manual_download_if_needed()

---

## 🎯 Resumen de cambios

| Antes | Después |
|-------|----------|
| Descarga siempre (9 min cada vez) | Descarga solo si falta (o si forzas) |
| Sin control de versión | Guarda timestamp y manifest de última descarga |
| Un solo notebook monolítico | Mismo notebook, pero inteligente |
| Comentario manual "no descargar" | Lógica automática de caché |

### ¿Un notebook o dos scripts?

**Respuesta: Uno es suficiente**, pero la estructura es:

```
01_download_kaggle_enemdu.ipynb  ← THIS (descarga + caché)
  ↓ (genera)
data/bronze/externas/enemdu/     ← caché local
  ↓ (Lee)
02_processing_silver.ipynb        ← transformación (Silver + Gold)
```

**No** necesitas dos notebooks de descarga. Lo que necesitas es:
1. **Este notebook** (con caché inteligente) — se ejecuta 1 vez o cuando hay nuevos datos
2. **Otro notebook** (Silver + Gold) — se ejecuta siempre, lee del caché

O, si prefieres **máxima eficiencia**, crea un script CLI:

```bash
# Solo descargar si falta
python scripts/ensure_bronze.py

# Luego procesar (Silver + Gold)
python scripts/run_enemdu_pipeline.py
```